In [ ]:
# ---- Install ONNX tooling (Internet ON) ----
!pip install -q onnx onnxruntime onnxscript


In [ ]:
%%writefile /kaggle/working/qat_job.py
"""
Quantization-aware fine-tuning of one final checkpoint (one process = one GPU).
Fake INT8 quantization is inserted where ONNX Runtime's static QDQ quantization will round:
  - every conv input (per-tensor uint8, asymmetric, min/max range)
  - every BatchNorm output before its activation (= the folded conv output in ONNX)
  - outputs of convs without BatchNorm (squeeze-excitation convs)
  - conv weights (per-channel symmetric int8)
BatchNorm statistics are frozen so training matches the folded inference graph.
Output: <tag>_qat.pt (plain FP32 weights, quantization-friendly) + <tag>_qat.json (history).
"""
import argparse, glob, json, math, os, random, time
import numpy as np, pandas as pd, timm, torch, torch.nn as nn
import torch.nn.utils.parametrize as P
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

IN, OUT, IMG = "/kaggle/input", "/kaggle/working", 224


def find_image_root(sample):
    if os.path.exists(os.path.join(IN, sample)):
        return IN
    first = sample.split("/")[0]
    for d in glob.glob(f"{IN}/**/{first}", recursive=True):
        if os.path.exists(os.path.join(os.path.dirname(d), sample)):
            return os.path.dirname(d)
    raise FileNotFoundError("Attach the PlantVillage dataset.")


class LeafDataset(Dataset):
    def __init__(self, df, tf, root):
        self.paths = [os.path.join(root, p) for p in df["path"]]; self.labels = df["label_idx"].tolist(); self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]


class ActFQ:
    """Per-tensor uint8 fake quantization with a min/max range (like ORT MinMax calibration)."""
    def __init__(self): self.lo = self.hi = None
    def reset(self): self.lo = self.hi = None
    def __call__(self, x, training):
        if training:
            lo, hi = x.detach().min().clamp(max=0), x.detach().max().clamp(min=0)
            self.lo = lo if self.lo is None else torch.minimum(self.lo, lo)
            self.hi = hi if self.hi is None else torch.maximum(self.hi, hi)
        if self.lo is None:
            return x
        scale = float(max((self.hi - self.lo) / 255.0, 1e-8)); zp = min(max(int(round(float(-self.lo / scale))), 0), 255)
        return torch.fake_quantize_per_tensor_affine(x, scale, zp, 0, 255)


class WeightFQ(nn.Module):
    def forward(self, w):
        s = (w.detach().abs().amax(dim=tuple(range(1, w.dim()))) / 127).clamp(min=1e-8)
        return torch.fake_quantize_per_channel_affine(w, s, torch.zeros_like(s, dtype=torch.int32), 0, -127, 127)


def attach_fake_quant(model):
    fqs, handles = [], []
    for mod in model.modules():
        if isinstance(mod, nn.Conv2d):
            fq = ActFQ(); fqs.append(fq)
            handles.append(mod.register_forward_pre_hook(lambda m, i, fq=fq: (fq(i[0], m.training),)))
            P.register_parametrization(mod, "weight", WeightFQ())
            if mod.bias is not None:  # conv not followed by BN: its output is quantized too
                fq2 = ActFQ(); fqs.append(fq2)
                handles.append(mod.register_forward_hook(lambda m, i, o, fq=fq2: fq(o, m.training)))
        if isinstance(mod, nn.modules.batchnorm._BatchNorm) and hasattr(mod, "act"):  # timm BatchNormAct2d
            fq = ActFQ(); fqs.append(fq)
            handles.append(mod.act.register_forward_pre_hook(lambda m, i, fq=fq: (fq(i[0], m.training),)))
    return fqs, handles


def detach_fake_quant(model, handles):
    for h in handles:
        h.remove()
    for mod in model.modules():
        if isinstance(mod, nn.Conv2d) and P.is_parametrized(mod, "weight"):
            P.remove_parametrizations(mod, "weight", leave_parametrized=False)


def freeze_bn(model):
    for mod in model.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm):
            mod.eval()


@torch.no_grad()
def evaluate(model, loader, dev):
    model.eval(); preds, labels = [], []
    for x, y in loader:
        preds.append(model(x.to(dev, non_blocking=True)).argmax(1).cpu()); labels.append(y)
    p, l = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    return {"acc": float((p == l).mean()), "macro_f1": float(f1_score(l, p, average="macro"))}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True); ap.add_argument("--ckpt", required=True); ap.add_argument("--tag", required=True)
    ap.add_argument("--epochs", type=int, default=3); ap.add_argument("--lr", type=float, default=5e-5)
    ap.add_argument("--bs", type=int, default=64); ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--reset-every", type=int, default=200); ap.add_argument("--max-steps", type=int, default=0)
    a = ap.parse_args()
    torch.manual_seed(0); random.seed(0); np.random.seed(0)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    df = pd.read_csv(glob.glob(f"{IN}/**/splits.csv", recursive=True)[0]); root = find_image_root(df["path"].iloc[0])
    cfg = timm.models.get_pretrained_cfg(a.model).to_dict(); norm = transforms.Normalize(cfg["mean"], cfg["std"])
    train_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                                   transforms.RandomRotation(10), transforms.ColorJitter(brightness=(0.5, 1.5)), transforms.ToTensor(), norm])
    eval_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), norm])
    kw = dict(num_workers=a.workers, pin_memory=True, persistent_workers=a.workers > 0)
    tl = DataLoader(LeafDataset(df[df.split == "train"], train_tf, root), batch_size=a.bs, shuffle=True, drop_last=True, **kw)
    vl = DataLoader(LeafDataset(df[df.split == "val"], eval_tf, root), batch_size=128, **kw)

    model = timm.create_model(a.model, pretrained=False, num_classes=df["label_idx"].nunique())
    model.load_state_dict(torch.load(a.ckpt, map_location="cpu")); model.to(dev)
    fqs, handles = attach_fake_quant(model)
    print(f"[{a.tag}] {dev} | {len(fqs)} fake-quant points | {a.epochs} epochs, lr {a.lr}", flush=True)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=0.0)
    total = a.epochs * len(tl) if not a.max_steps else a.max_steps
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * min(s, total) / total)))
    crit = nn.CrossEntropyLoss(); hist, step, t0 = [], 0, time.time()
    for ep in range(1, a.epochs + 1):
        model.train(); freeze_bn(model); run, n = 0.0, 0
        for x, y in tl:
            if step % a.reset_every == 0:  # ranges follow the current weights
                for fq in fqs: fq.reset()
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            loss = crit(model(x), y); opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
            run += loss.item() * len(y); n += len(y); step += 1
            if a.max_steps and step >= a.max_steps: break
        val = evaluate(model, vl, dev)  # with simulated INT8 (hooks active)
        hist.append({"epoch": ep, "train_loss": run / n, "val_sim_int8_acc": val["acc"], "val_sim_int8_f1": val["macro_f1"],
                     "time_s": time.time() - t0})
        print(f"[{a.tag}] " + json.dumps(hist[-1]), flush=True)
        if a.max_steps and step >= a.max_steps: break
    detach_fake_quant(model, handles)
    fp32_val = evaluate(model, vl, dev)
    torch.save(model.state_dict(), f"{OUT}/{a.tag}_qat.pt")
    json.dump({"tag": a.tag, "model": a.model, "epochs": a.epochs, "lr": a.lr, "history": hist, "val_fp32_after_qat": fp32_val},
              open(f"{OUT}/{a.tag}_qat.json", "w"), indent=2)
    print(f"[{a.tag}] DONE fp32 val after QAT {fp32_val}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# ---- Stage 1 (GPU): QAT fine-tuning of all 6 final checkpoints, one job per GPU at a time ----
import glob, json, os, queue, subprocess, sys, threading, time
import torch

QAT_EPOCHS = 3
RUNS = {"mobilenetv3_small_100": ["0", "1_full", "2"], "mobilevit_xxs": ["0", "1", "2"]}
CKPT = {}
for model, seeds in RUNS.items():
    for s in seeds:
        hits = glob.glob(f"/kaggle/input/**/{model}_final_seed{s}.pt", recursive=True)
        assert hits, f"{model}_final_seed{s}.pt not found - add the Phase 2 and rerun notebook outputs as inputs."
        CKPT[(model, s)] = hits[0]
n_gpus = max(torch.cuda.device_count(), 1)
workers = 4 if n_gpus == 1 else 2
print(f"{n_gpus} GPU(s); {len(CKPT)} checkpoints", flush=True)

q = queue.Queue()
for (m, s), p in CKPT.items():
    q.put((m, s, p))
failed = []

def gpu_worker(g):
    while True:
        try:
            m, s, p = q.get_nowait()
        except queue.Empty:
            return
        tag = f"{m}_seed{s[0]}"
        if os.path.exists(f"/kaggle/working/{tag}_qat.pt"):
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        proc = subprocess.Popen([sys.executable, "-u", "/kaggle/working/qat_job.py", "--model", m, "--ckpt", p, "--tag", tag,
                                 "--epochs", str(QAT_EPOCHS), "--workers", str(workers)],
                                env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            if "Warning" not in line and "warn(" not in line:
                print(f"[gpu{g}] {line}", end="", flush=True)
        if proc.wait() != 0:
            failed.append(tag)

threads = [threading.Thread(target=gpu_worker, args=(g,)) for g in range(n_gpus)]
for t in threads: t.start()
for t in threads: t.join()
assert not failed, f"QAT failed for {failed}"


In [ ]:
# ---- Stage 2 (CPU): ONNX export, INT8 quantization (full and mixed precision), test-set evaluation ----
import logging, warnings
logging.getLogger().setLevel(logging.ERROR); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, timm, onnx, onnxruntime as ort, zipfile
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from sklearn.metrics import f1_score
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static
from onnxruntime.quantization.shape_inference import quant_pre_process

OUT = "/kaggle/working"; ONNX_DIR = f"{OUT}/onnx"; os.makedirs(ONNX_DIR, exist_ok=True)
NT = os.cpu_count(); IMG = 224
OPS = ["Conv", "Gemm", "MatMul"]   # quantize compute layers only (quantizing Add/Mul/activations made things worse)
df = pd.read_csv(glob.glob("/kaggle/input/**/splits.csv", recursive=True)[0])
first = df["path"].iloc[0].split("/")[0]
ROOT = next(os.path.dirname(d) for d in glob.glob(f"/kaggle/input/**/{first}", recursive=True)
            if os.path.exists(os.path.join(os.path.dirname(d), df["path"].iloc[0])))

def load_u8(paths):
    one = lambda p: np.asarray(Image.open(os.path.join(ROOT, p)).convert("RGB").resize((IMG, IMG), Image.BILINEAR))
    with ThreadPoolExecutor(NT) as ex:
        return np.stack(list(ex.map(one, paths)))

test = df[df.split == "test"]; X_TEST, Y_TEST = load_u8(test["path"].tolist()), test["label_idx"].to_numpy()
val_s = df[df.split == "val"].sample(min(1000, (df.split == "val").sum()), random_state=0); X_VAL, Y_VAL = load_u8(val_s["path"].tolist()), val_s["label_idx"].to_numpy()
cal = df[df.split == "train"].groupby("label_idx", group_keys=False).apply(lambda g: g.sample(min(len(g), 7), random_state=0))
X_CAL = load_u8(cal["path"].tolist())
print(f"test {len(X_TEST)}, val subset {len(X_VAL)} (for layer selection), calibration {len(X_CAL)}", flush=True)

def to_input(u8, model):
    c = timm.models.get_pretrained_cfg(model).to_dict()
    return ((u8.astype(np.float32) / 255).transpose(0, 3, 1, 2) - np.array(c["mean"], np.float32).reshape(1, 3, 1, 1)) \
        / np.array(c["std"], np.float32).reshape(1, 3, 1, 1)

class CalibReader(CalibrationDataReader):
    def __init__(self, x): self.it = iter([{"input": x[i:i + 16]} for i in range(0, len(x), 16)])
    def get_next(self): return next(self.it, None)

def predict(path, x):
    so = ort.SessionOptions(); so.intra_op_num_threads = NT
    s = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
    return np.concatenate([s.run(None, {"input": x[i:i + 64]})[0].argmax(1) for i in range(0, len(x), 64)])

def export(model_name, state_path, out):
    m = timm.create_model(model_name, pretrained=False, num_classes=df["label_idx"].nunique()); m.load_state_dict(torch.load(state_path, map_location="cpu")); m.eval()
    kw = dict(input_names=["input"], output_names=["logits"], opset_version=17, dynamic_axes={"input": {0: "b"}, "logits": {0: "b"}})
    try:
        torch.onnx.export(m, torch.randn(1, 3, IMG, IMG), out, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(m, torch.randn(1, 3, IMG, IMG), out, **kw)
    pre = out.replace(".onnx", "_pre.onnx"); quant_pre_process(out, pre); return pre

def quantize(pre, out, x_cal, exclude=()):
    quantize_static(pre, out, CalibReader(x_cal), quant_format=QuantFormat.QDQ, per_channel=True, op_types_to_quantize=OPS,
                    activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8, nodes_to_exclude=list(exclude))
    return out

def rank_layers(pre, x_cal, x_val, y_val):
    """Quantize one conv at a time; rank convs by the validation accuracy that this single conv costs."""
    convs = [n.name for n in onnx.load(pre).graph.node if n.op_type == "Conv"]
    tmp = f"{ONNX_DIR}/_one.onnx"; scores = []
    for c in convs:
        quantize_static(pre, tmp, CalibReader(x_cal[:128]), quant_format=QuantFormat.QDQ, per_channel=True, op_types_to_quantize=["Conv"],
                        nodes_to_quantize=[c])
        scores.append((float((predict(tmp, x_val) == y_val).mean()), c))
    return [c for _, c in sorted(scores)], len(convs)

KS = [0, 2, 4, 8, 12, 16, 24, 32]
rows, chosen = [], {}
for model_name, seeds in RUNS.items():
    for kind in ["ptq", "qat"]:     # ptq = original fine-tuned weights, qat = after quantization-aware fine-tuning
        for s in seeds:
            tag = f"{model_name}_seed{s[0]}"
            state = CKPT[(model_name, s)] if kind == "ptq" else f"{OUT}/{tag}_qat.pt"
            pre = export(model_name, state, f"{ONNX_DIR}/{tag}_{kind}_fp32.onnx")
            x_cal, x_val, x_test = to_input(X_CAL, model_name), to_input(X_VAL, model_name), to_input(X_TEST, model_name)
            if s[0] == "0":   # choose how many sensitive convs stay FP32, on the validation subset, seed 0 only
                ranked, n_conv = rank_layers(pre, x_cal, x_val, Y_VAL)
                fp32_val = float((predict(pre, x_val) == Y_VAL).mean()); sweep = []
                for k in [k for k in KS if k <= n_conv]:
                    acc = float((predict(quantize(pre, f"{ONNX_DIR}/_k.onnx", x_cal, ranked[:k]), x_val) == Y_VAL).mean())
                    sweep.append({"k": k, "val_acc": acc}); print(f"  {model_name} {kind}: keep {k:2d}/{n_conv} convs FP32 -> val INT8 acc {acc:.4f} (FP32 {fp32_val:.4f})", flush=True)
                ok = [r["k"] for r in sweep if r["val_acc"] >= fp32_val - 0.01]
                k_sel = ok[0] if ok else sweep[-1]["k"]
                chosen[(model_name, kind)] = {"k": k_sel, "excluded": ranked[:k_sel], "n_conv": n_conv, "sweep": sweep, "fp32_val": fp32_val}
                print(f"  -> {model_name} {kind}: keep {k_sel} convs in FP32", flush=True)
            sel = chosen[(model_name, kind)]
            variants = {"fp32": pre,
                        "int8_full": quantize(pre, f"{ONNX_DIR}/{tag}_{kind}_int8_full.onnx", x_cal),
                        "int8_mixed": quantize(pre, f"{ONNX_DIR}/{tag}_{kind}_int8_mixed.onnx", x_cal, sel["excluded"])}
            ref = None
            for v, p in variants.items():
                pr = predict(p, x_test); ref = pr if v == "fp32" else ref
                rows.append({"model": model_name, "weights": kind, "seed": int(s[0]), "variant": v, "fp32_convs_kept": sel["k"] if v == "int8_mixed" else (0 if v == "int8_full" else None),
                             "size_MB": os.path.getsize(p) / 2**20, "test_acc": float((pr == Y_TEST).mean()),
                             "test_macro_f1": float(f1_score(Y_TEST, pr, average="macro")), "agree_with_fp32": float((pr == ref).mean())})
                print(f"{tag} {kind:3s} {v:10s} {rows[-1]['size_MB']:5.2f} MB  test acc {rows[-1]['test_acc']:.4f}  F1 {rows[-1]['test_macro_f1']:.4f}", flush=True)
            os.remove(pre) if s[0] != "0" else None

res = pd.DataFrame(rows); res.to_csv(f"{OUT}/qat_results.csv", index=False)
json.dump({f"{m}|{k}": v for (m, k), v in chosen.items()}, open(f"{OUT}/mixed_precision_selection.json", "w"), indent=2)
summary = res.groupby(["model", "weights", "variant"]).agg(size_MB=("size_MB", "mean"), acc_mean=("test_acc", "mean"),
           acc_std=("test_acc", "std"), f1_mean=("test_macro_f1", "mean"), fp32_convs_kept=("fp32_convs_kept", "first")).round(4)
summary.to_csv(f"{OUT}/qat_summary.csv"); print(summary.to_string())

# Raspberry Pi bundle 2: seed-0 files of every variant (copy into pi_bundle/onnx/ next to the first bundle)
with zipfile.ZipFile(f"{OUT}/pi_bundle_qat.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{ONNX_DIR}/*_seed0_*.onnx"):
        if not f.endswith("_pre.onnx") and "/_" not in f:
            z.write(f, os.path.basename(f))
print("Saved qat_results.csv, qat_summary.csv, mixed_precision_selection.json, pi_bundle_qat.zip")
